# Medical Insurance Cost Prediction
**Student Name:** Nadeem Khan  
**Project Type:** Machine Learning - Regression  
**Dataset:** Medical Insurance Cost Prediction.csv  

---
## Objective
Predict medical insurance charges based on customer features (age, BMI, smoking status, children, sex, region) using supervised ML models.

---

## Section 1: Import Libraries

In [1]:
# Import all necessary libraries
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print('All libraries imported successfully!')

All libraries imported successfully!


## Section 2: Load and Understand the Dataset

In [2]:
# Load the dataset
df = pd.read_csv('Medical Insurance Cost Prediction.csv')
print('Dataset shape:', df.shape)
print('\nFirst 5 rows:')
df.head()

Dataset shape: (1338, 7)

First 5 rows:


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [3]:
# Check column data types and non-null counts
print('Dataset Info:')
df.info()

Dataset Info:
<class 'pandas.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   str    
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   str    
 5   region    1338 non-null   str    
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), str(3)
memory usage: 73.3 KB


In [4]:
# Statistical summary of numerical columns
print('Statistical Summary:')
df.describe()

Statistical Summary:


,age,bmi,children,charges
count,1338.000000,1338.000000,1338.000000,1338.000000
mean,39.207025,30.663397,1.094918,13270.422265
std,14.049960,6.098187,1.205493,12110.011237
min,18.000000,15.960000,0.000000,1121.873900
25%,27.000000,26.296250,0.000000,4740.287150
50%,39.000000,30.400000,1.000000,9382.033000
75%,51.000000,34.693750,2.000000,16639.912515
max,64.000000,53.130000,5.000000,63770.428010


## Section 3: Data Quality Check

In [5]:
# Check for missing values
print('Missing Values per Column:')
print(df.isnull().sum())

# Check for duplicate rows
duplicates = df.duplicated().sum()
print(f'\nNumber of duplicate rows: {duplicates}')

Missing Values per Column:
age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

Number of duplicate rows: 1


In [6]:
# Remove duplicates if any
if duplicates > 0:
    print('Duplicate row(s):')
    print(df[df.duplicated(keep=False)])
    df = df.drop_duplicates().reset_index(drop=True)
    print(f'\nAfter removing duplicates, dataset shape: {df.shape}')
else:
    print('No duplicates found.')

Duplicate row(s):
     age   sex    bmi  children smoker     region    charges
195   19  male  30.59         0     no  northwest  1639.5631
581   19  male  30.59         0     no  northwest  1639.5631

After removing duplicates, dataset shape: (1337, 7)


In [7]:
# Check unique values for categorical columns
for col in ['sex', 'smoker', 'region']:
    print(f'{col}: {df[col].unique()}')

sex: <StringArray>
['female', 'male']
Length: 2, dtype: str
smoker: <StringArray>
['yes', 'no']
Length: 2, dtype: str
region: <StringArray>
['southwest', 'southeast', 'northwest', 'northeast']
Length: 4, dtype: str


In [8]:
# Check valid ranges for numerical columns
print('Age range:', df['age'].min(), '-', df['age'].max())
print('BMI range:', df['bmi'].min(), '-', df['bmi'].max())
print('Children range:', df['children'].min(), '-', df['children'].max())
print('Charges range:', df['charges'].min(), '-', df['charges'].max())

Age range: 18 - 64
BMI range: 15.96 - 53.13
Children range: 0 - 5
Charges range: 1121.8739 - 63770.42801


## Section 4: Exploratory Data Analysis (EDA)

In [9]:
# --- Chart 1: Distribution of Insurance Charges ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['charges'], bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of Insurance Charges', fontsize=14)
axes[0].set_xlabel('Charges ($)')
axes[0].set_ylabel('Frequency')

axes[1].hist(np.log1p(df['charges']), bins=40, color='darkorange', edgecolor='white')
axes[1].set_title('Log-Transformed Distribution of Charges', fontsize=14)
axes[1].set_xlabel('Log(Charges)')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.savefig('chart_charges_distribution.png', dpi=100, bbox_inches='tight')
plt.show()
print('Chart 1 saved.')

Chart 1 saved.


In [10]:
# --- Chart 2: Charges by Smoker Status (Boxplot) ---
# Note: matplotlib>=3.9 uses set_xticklabels instead of 'labels' parameter
fig, ax = plt.subplots(figsize=(8, 5))
smoker_groups = [df[df['smoker']=='yes']['charges'].values,
                 df[df['smoker']=='no']['charges'].values]
ax.boxplot(smoker_groups, patch_artist=True,
           boxprops=dict(facecolor='tomato', color='black'),
           medianprops=dict(color='black', linewidth=2))
ax.set_xticks([1, 2])
ax.set_xticklabels(['Smoker', 'Non-Smoker'])
ax.set_title('Insurance Charges: Smoker vs Non-Smoker', fontsize=14)
ax.set_ylabel('Charges ($)')
plt.tight_layout()
plt.savefig('chart_smoker_charges.png', dpi=100, bbox_inches='tight')
plt.show()
print('Chart 2 saved.')

Chart 2 saved.


In [11]:
# --- Chart 3: Age vs Charges (coloured by smoker) ---
colors = df['smoker'].map({'yes': 'tomato', 'no': 'steelblue'})
fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(df['age'], df['charges'], c=colors, alpha=0.5, edgecolors='none', s=30)
ax.set_title('Age vs Insurance Charges (Red=Smoker, Blue=Non-Smoker)', fontsize=13)
ax.set_xlabel('Age')
ax.set_ylabel('Charges ($)')
plt.tight_layout()
plt.savefig('chart_age_vs_charges.png', dpi=100, bbox_inches='tight')
plt.show()
print('Chart 3 saved.')

Chart 3 saved.


In [12]:
# --- Chart 4: BMI vs Charges ---
fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(df['bmi'], df['charges'], c=colors, alpha=0.5, edgecolors='none', s=30)
ax.set_title('BMI vs Insurance Charges (Red=Smoker, Blue=Non-Smoker)', fontsize=13)
ax.set_xlabel('BMI')
ax.set_ylabel('Charges ($)')
plt.tight_layout()
plt.savefig('chart_bmi_vs_charges.png', dpi=100, bbox_inches='tight')
plt.show()
print('Chart 4 saved.')

Chart 4 saved.


In [13]:
# --- Chart 5: Average Charges by Region ---
region_avg = df.groupby('region')['charges'].mean().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(region_avg.index, region_avg.values,
              color=['#4e79a7','#f28e2b','#e15759','#76b7b2'], edgecolor='white')
ax.set_title('Average Insurance Charges by Region', fontsize=14)
ax.set_xlabel('Region')
ax.set_ylabel('Average Charges ($)')
for bar, val in zip(bars, region_avg.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
            f'${val:,.0f}', ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.savefig('chart_region_charges.png', dpi=100, bbox_inches='tight')
plt.show()
print('Chart 5 saved.')

Chart 5 saved.


In [14]:
# --- Chart 6: Correlation Heatmap ---
df_corr = df.copy()
df_corr['smoker_enc'] = (df_corr['smoker'] == 'yes').astype(int)
corr_cols = ['age', 'bmi', 'children', 'smoker_enc', 'charges']
corr_matrix = df_corr[corr_cols].corr()

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(corr_matrix.values, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)
ax.set_xticks(range(len(corr_cols)))
ax.set_yticks(range(len(corr_cols)))
labels = ['Age', 'BMI', 'Children', 'Smoker', 'Charges']
ax.set_xticklabels(labels, rotation=45, ha='right')
ax.set_yticklabels(labels)
for i in range(len(corr_cols)):
    for j in range(len(corr_cols)):
        val = corr_matrix.values[i, j]
        txt_color = 'white' if abs(val) > 0.5 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                fontsize=10, color=txt_color)
ax.set_title('Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.savefig('chart_correlation_heatmap.png', dpi=100, bbox_inches='tight')
plt.show()
print('Chart 6 saved.')
print('\nCorrelation with Charges:')
print(corr_matrix['charges'].sort_values(ascending=False))

Chart 6 saved.

Correlation with Charges:
charges       1.000000
smoker_enc    0.787234
age           0.298308
bmi           0.198401
children      0.067389
Name: charges, dtype: float64


In [15]:
# --- Chart 7: Charges by Number of Children ---
children_avg = df.groupby('children')['charges'].mean()
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(children_avg.index.astype(str), children_avg.values,
       color='mediumseagreen', edgecolor='white')
ax.set_title('Average Insurance Charges by Number of Children', fontsize=14)
ax.set_xlabel('Number of Children')
ax.set_ylabel('Average Charges ($)')
plt.tight_layout()
plt.savefig('chart_children_charges.png', dpi=100, bbox_inches='tight')
plt.show()
print('Chart 7 saved.')

Chart 7 saved.


## Section 5: Feature Engineering & Preprocessing

In [16]:
# Create a copy for preprocessing (keep original df untouched)
df_model = df.copy()

# Encode binary categorical features with LabelEncoder
le = LabelEncoder()
df_model['sex']    = le.fit_transform(df_model['sex'])    # female=0, male=1
df_model['smoker'] = le.fit_transform(df_model['smoker']) # no=0, yes=1

# One-hot encode 'region' (drop first to avoid multicollinearity)
df_model = pd.get_dummies(df_model, columns=['region'], drop_first=True)

print('Encoded dataset shape:', df_model.shape)
print('Columns:', df_model.columns.tolist())
df_model.head()

Encoded dataset shape: (1337, 9)
Columns: ['age', 'sex', 'bmi', 'children', 'smoker', 'charges', 'region_northwest', 'region_southeast', 'region_southwest']


,age,sex,bmi,children,smoker,charges,region_northwest,region_southeast,region_southwest
0,19,0,27.900,0,1,16884.92400,False,False,True
1,18,1,33.770,1,0,1725.55230,False,True,False
2,28,1,33.000,3,0,4449.46200,False,True,False
3,33,1,22.705,0,0,21984.47061,True,False,False
4,32,1,28.880,0,0,3866.85520,True,False,False


In [17]:
# Separate features (X) and target (y)
X = df_model.drop('charges', axis=1)
y = df_model['charges']

print('Feature matrix shape:', X.shape)
print('Target vector shape: ', y.shape)

Feature matrix shape: (1337, 8)
Target vector shape:  (1337,)


In [18]:
# Split into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f'Training set size: {X_train.shape[0]} rows')
print(f'Testing set size:  {X_test.shape[0]} rows')

Training set size: 1069 rows
Testing set size:  268 rows


In [19]:
# Scale numerical features for Linear Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)
print('Scaling complete.')

Scaling complete.


## Section 6: Model Training

In [20]:
# --- Model 1: Linear Regression ---
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)
print('Linear Regression model trained successfully!')

Linear Regression model trained successfully!


In [21]:
# --- Model 2: Random Forest Regressor ---
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)  # RF uses unscaled data
print('Random Forest Regressor model trained successfully!')

Random Forest Regressor model trained successfully!


## Section 7: Model Evaluation

In [22]:
# Helper function to evaluate a model
def evaluate_model(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    print(f'--- {name} ---')
    print(f'  MAE  : ${mae:,.2f}')
    print(f'  RMSE : ${rmse:,.2f}')
    print(f'  R2   : {r2:.4f}')
    return mae, rmse, r2

# Predictions
lr_pred = lr_model.predict(X_test_scaled)
rf_pred = rf_model.predict(X_test)

# Evaluate both models
lr_mae, lr_rmse, lr_r2 = evaluate_model('Linear Regression', y_test, lr_pred)
print()
rf_mae, rf_rmse, rf_r2 = evaluate_model('Random Forest Regressor', y_test, rf_pred)

--- Linear Regression ---
  MAE  : $4,177.05
  RMSE : $5,956.34
  R2   : 0.8069

--- Random Forest Regressor ---
  MAE  : $2,632.78
  RMSE : $4,697.78
  R2   : 0.8799


In [23]:
# --- Chart 8: Model Comparison Bar Chart ---
metrics   = ['MAE', 'RMSE', 'R2 Score']
lr_scores = [lr_mae, lr_rmse, lr_r2]
rf_scores = [rf_mae, rf_rmse, rf_r2]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, (metric, lr_val, rf_val) in enumerate(zip(metrics, lr_scores, rf_scores)):
    axes[i].bar(['Linear\nRegression', 'Random\nForest'], [lr_val, rf_val],
                color=['#4e79a7', '#f28e2b'], edgecolor='white', width=0.5)
    axes[i].set_title(metric, fontsize=13)
    axes[i].set_ylabel(metric)
    for j, val in enumerate([lr_val, rf_val]):
        label = f'{val:.4f}' if metric == 'R2 Score' else f'${val:,.0f}'
        axes[i].text(j, val * 0.97, label, ha='center', va='top',
                     fontsize=10, color='white', fontweight='bold')
plt.suptitle('Model Performance Comparison', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('chart_model_comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print('Chart 8 saved.')

Chart 8 saved.


In [24]:
# --- Chart 9: Actual vs Predicted ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, pred, name, color in zip(
        axes, [lr_pred, rf_pred],
        ['Linear Regression', 'Random Forest'],
        ['steelblue', 'darkorange']):
    ax.scatter(y_test, pred, alpha=0.4, color=color, edgecolors='none', s=25)
    min_val = min(float(y_test.min()), float(pred.min()))
    max_val = max(float(y_test.max()), float(pred.max()))
    ax.plot([min_val, max_val], [min_val, max_val], 'k--', lw=1.5, label='Perfect Fit')
    ax.set_title(f'{name}: Actual vs Predicted', fontsize=13)
    ax.set_xlabel('Actual Charges ($)')
    ax.set_ylabel('Predicted Charges ($)')
    ax.legend()
plt.tight_layout()
plt.savefig('chart_actual_vs_predicted.png', dpi=100, bbox_inches='tight')
plt.show()
print('Chart 9 saved.')

Chart 9 saved.


In [25]:
# --- Chart 10: Feature Importance (Random Forest) ---
feature_names = X.columns.tolist()
importances   = rf_model.feature_importances_
sorted_idx    = np.argsort(importances)[::-1]

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(range(len(feature_names)), importances[sorted_idx],
       color='mediumpurple', edgecolor='white')
ax.set_xticks(range(len(feature_names)))
ax.set_xticklabels([feature_names[i] for i in sorted_idx], rotation=30, ha='right')
ax.set_title('Feature Importances - Random Forest', fontsize=14)
ax.set_ylabel('Importance Score')
plt.tight_layout()
plt.savefig('chart_feature_importance.png', dpi=100, bbox_inches='tight')
plt.show()
print('Chart 10 saved.')
print('\nFeature Importances:')
for i in sorted_idx:
    print(f'  {feature_names[i]:<25} {importances[i]:.4f}')

Chart 10 saved.

Feature Importances:
  smoker                    0.6008
  bmi                       0.2164
  age                       0.1356
  children                  0.0229
  sex                       0.0070
  region_northwest          0.0067
  region_southeast          0.0061
  region_southwest          0.0046


## Section 8: Results Summary

In [26]:
# Print a clean comparison table
print('=' * 55)
print(f"{'Model':<28} {'MAE':>8} {'RMSE':>10} {'R2':>6}")
print('-' * 55)
print(f"{'Linear Regression':<28} ${lr_mae:>7,.0f} ${lr_rmse:>9,.0f} {lr_r2:>6.4f}")
print(f"{'Random Forest Regressor':<28} ${rf_mae:>7,.0f} ${rf_rmse:>9,.0f} {rf_r2:>6.4f}")
print('=' * 55)
best = 'Random Forest Regressor' if rf_r2 > lr_r2 else 'Linear Regression'
print(f'\nBest Performing Model: {best}')

Model                             MAE       RMSE     R2
-------------------------------------------------------
Linear Regression            $  4,177 $    5,956 0.8069
Random Forest Regressor      $  2,633 $    4,698 0.8799

Best Performing Model: Random Forest Regressor


## Section 9: Business Insights & Conclusions

In [27]:
# Compute key statistics for insights
smoker_mean    = df[df['smoker']=='yes']['charges'].mean()
nonsmoker_mean = df[df['smoker']=='no']['charges'].mean()
smoker_ratio   = smoker_mean / nonsmoker_mean

corr_age    = df_corr['age'].corr(df_corr['charges'])
corr_bmi    = df_corr['bmi'].corr(df_corr['charges'])
corr_smoker = df_corr['smoker_enc'].corr(df_corr['charges'])

region_avg_sorted = df.groupby('region')['charges'].mean().sort_values(ascending=False)

print('KEY BUSINESS INSIGHTS')
print('=' * 55)
print('1. Smoker Impact:')
print(f'   Average charges (Smoker)    : ${smoker_mean:>10,.2f}')
print(f'   Average charges (Non-Smoker): ${nonsmoker_mean:>10,.2f}')
print(f'   Smokers pay ~{smoker_ratio:.1f}x more in insurance costs')
print()
print('2. Feature Correlations with Charges:')
print(f'   Smoker  : {corr_smoker:.4f}  (strongest predictor)')
print(f'   Age     : {corr_age:.4f}')
print(f'   BMI     : {corr_bmi:.4f}')
print()
print('3. Region-wise Average Charges:')
for region, val in region_avg_sorted.items():
    print(f'   {region:<12}: ${val:>10,.2f}')
print()
print('4. Model Performance:')
print(f'   Random Forest R2 = {rf_r2:.4f} -> explains {rf_r2*100:.1f}% of charge variance')
print(f'   Linear Regression R2 = {lr_r2:.4f} -> explains {lr_r2*100:.1f}% of charge variance')

KEY BUSINESS INSIGHTS
1. Smoker Impact:
   Average charges (Smoker)    : $ 32,050.23
   Average charges (Non-Smoker): $  8,440.66
   Smokers pay ~3.8x more in insurance costs

2. Feature Correlations with Charges:
   Smoker  : 0.7872  (strongest predictor)
   Age     : 0.2983
   BMI     : 0.1984

3. Region-wise Average Charges:
   southeast   : $ 14,735.41
   northeast   : $ 13,406.38
   northwest   : $ 12,450.84
   southwest   : $ 12,346.94

4. Model Performance:
   Random Forest R2 = 0.8799 -> explains 88.0% of charge variance
   Linear Regression R2 = 0.8069 -> explains 80.7% of charge variance


In [28]:
conclusions = '''
CONCLUSIONS
===========
1. Smoking is by far the strongest predictor of medical insurance charges,
   with smokers paying roughly 3.8x more than non-smokers.

2. Age and BMI are the next most influential numerical features;
   older and higher-BMI individuals consistently have higher charges.

3. Region has a modest effect; the southeast region carries
   the highest average charges among all four regions.

4. Random Forest Regressor outperforms Linear Regression on all metrics
   (lower MAE, lower RMSE, and higher R2), because it captures non-linear
   relationships in the data that a linear model cannot model.

5. This project demonstrates a complete end-to-end ML workflow:
   data cleaning -> EDA -> preprocessing -> model training -> evaluation.
   The Random Forest model is recommended for deployment.
'''
print(conclusions)


CONCLUSIONS
1. Smoking is by far the strongest predictor of medical insurance charges,
   with smokers paying roughly 3.8x more than non-smokers.

2. Age and BMI are the next most influential numerical features;
   older and higher-BMI individuals consistently have higher charges.

3. Region has a modest effect; the southeast region carries
   the highest average charges among all four regions.

4. Random Forest Regressor outperforms Linear Regression on all metrics
   (lower MAE, lower RMSE, and higher R2), because it captures non-linear
   relationships in the data that a linear model cannot model.

5. This project demonstrates a complete end-to-end ML workflow:
   data cleaning -> EDA -> preprocessing -> model training -> evaluation.
   The Random Forest model is recommended for deployment.



In [29]:
# Save final model metrics to text file
with open('model_results.txt', 'w') as f:
    f.write('Linear Regression:\n')
    f.write(f'  MAE  = {lr_mae:.4f}\n')
    f.write(f'  RMSE = {lr_rmse:.4f}\n')
    f.write(f'  R2   = {lr_r2:.4f}\n\n')
    f.write('Random Forest Regressor:\n')
    f.write(f'  MAE  = {rf_mae:.4f}\n')
    f.write(f'  RMSE = {rf_rmse:.4f}\n')
    f.write(f'  R2   = {rf_r2:.4f}\n')
print('model_results.txt saved.')
print('Project complete!')

model_results.txt saved.
Project complete!


---
## Project Complete
All sections executed successfully.

**Student:** Nadeem Khan | **Best Model:** Random Forest Regressor (R2 = 0.8799)